In [0]:
%run ./lib/00_common

In [0]:
%run ./lib/00_common

In [0]:
%run ./lib/02_delta_writer

In [0]:
%run ./lib/03_logger_utils

In [0]:
%run ./lib/06_alerting

In [0]:
import pandas as pd

run_date = get_text_widget("run_date", "2025-04-01")
run_id = get_or_create_run_id()

alert_email_to = get_text_widget("alert_email_to", "")
gmail_smtp_user = get_text_widget("gmail_smtp_user", "")
gmail_app_password = get_text_widget("gmail_app_password", "")

In [0]:

    stage = "bronze_ingest"
    dataset = "orders"

    try:
        
        #raise ValueError("¡ESTA ES UNA PRUEBA! Forzando el fallo para verificar el envío de correo.")

        log_event(
            spark=spark,
            level="INFO",
            run_id=run_id,
            stage=stage,
            dataset=dataset,
            status="STARTED",
            message=f"Inicia Bronze orders desde Volume con pandas para run_date={run_date}"
        )

        file_path = assert_file_exists(landing_orders_file(run_date))
        orders_pdf = pd.read_csv(file_path)

        orders_pdf["_ingestion_ts"] = utc_now_naive()
        orders_pdf["_source_file"] = file_path
        orders_pdf["_run_id"] = run_id
        orders_pdf["_run_date"] = run_date

        # 1. Transformar IDs a enteros que soportan nulos (BIGINT e INT)
        orders_pdf["order_id"] = orders_pdf["order_id"].astype("Int64")
        orders_pdf["customer_id"] = orders_pdf["customer_id"].astype("Int32")

        # 2. Transformar fechas (DATE)
        # Lo convertimos a datetime y luego extraemos solo la fecha
        orders_pdf["order_date"] = pd.to_datetime(orders_pdf["order_date"]).dt.date

        # 3. Asegurar los decimales (DOUBLE)
        orders_pdf["amount"] = orders_pdf["amount"].astype("float64")

        # 4. Asegurar el Timestamp de ingestión (TIMESTAMP)
        orders_pdf["_ingestion_ts"] = pd.to_datetime(orders_pdf["_ingestion_ts"])

        # 5. Transformar el resto a cadenas de texto explícitas (STRING)
        orders_pdf["status"] = orders_pdf["status"].astype("string")
        orders_pdf["_source_file"] = orders_pdf["_source_file"].astype("string")
        orders_pdf["_run_id"] = orders_pdf["_run_id"].astype("string")
        orders_pdf["_run_date"] = orders_pdf["_run_date"].astype("string")

        write_csv(orders_pdf, bronze_orders_snapshot_file(run_date))
        write_bronze_orders_delta(orders_pdf, mode="append")

        log_event(
            spark=spark,
            level="INFO",
            run_id=run_id,
            stage=stage,
            dataset=dataset,
            status="OK",
            message="Bronze orders completado",
            rows_ok=len(orders_pdf),
            extra={
                "landing_file": file_path,
                "bronze_snapshot_file": bronze_orders_snapshot_file(run_date),
                "run_date": run_date
            }
        )

    except Exception as e:
        notify_failure(
            spark=spark,
            run_id=run_id,
            stage=stage,
            dataset=dataset,
            error_message=str(e),
            error_code="BRZ_ORD_001",
            smtp_user=gmail_smtp_user,
            smtp_app_password=gmail_app_password,
            to_email=alert_email_to
        )
        raise


In [0]:
stage = "bronze_ingest"
dataset = "customers"

try:

    #raise ValueError("¡ESTA ES UNA PRUEBA! Forzando el fallo para verificar el envío de correo.")
    log_event(
        spark=spark,
        level="INFO",
        run_id=run_id,
        stage=stage,
        dataset=dataset,
        status="STARTED",
        message=f"Inicia Bronze customers desde Volume con pandas para run_date={run_date}"
    )

    file_path = assert_file_exists(landing_customers_file(run_date))
    customers_pdf = pd.read_json(file_path, lines=True)

    customers_pdf["_ingestion_ts"] = utc_now_naive()
    customers_pdf["_source_file"] = file_path
    customers_pdf["_run_id"] = run_id
    customers_pdf["_run_date"] = run_date

    write_csv(customers_pdf, bronze_customers_snapshot_file(run_date))
    write_bronze_customers_delta(customers_pdf, mode="append")

    log_event(
        spark=spark,
        level="INFO",
        run_id=run_id,
        stage=stage,
        dataset=dataset,
        status="OK",
        message="Bronze customers completado",
        rows_ok=len(customers_pdf),
        extra={
            "landing_file": file_path,
            "bronze_snapshot_file": bronze_customers_snapshot_file(run_date),
            "run_date": run_date
        }
    )

except Exception as e:
    notify_failure(
        spark=spark,
        run_id=run_id,
        stage=stage,
        dataset=dataset,
        error_message=str(e),
        error_code="BRZ_CUS_001",
        smtp_user=gmail_smtp_user,
        smtp_app_password=gmail_app_password,
        to_email=alert_email_to
    )
    raise

In [0]:
print(f"Bronze OK. run_id={run_id}")